# TradeSense · Component 03 · notebook 49: how reliable is an alert? Conformal sets and three evidence levels (protocol C1 v3, Parts 1 and 3; CPU, minutes)

**Question.** The engine says *what* is unusual. This notebook asks *how far to trust it*: (1) a conformal prediction set for every minute ("normal", "abnormal", "cannot tell"), calibrated on CAL-3 only and tested on the saved Sep/Oct scores; (2) three plain-language evidence levels for alerts (**Strong / Moderate / Limited**) built from three checks (rarity, persistence, no conflict between models) behind a data-quality gate. The conformal set and model agreement are shown but cannot create a level.

**What it does NOT do.** No model is retrained or changed; no threshold changes; the numbers describe the *score*, not whether manipulation happened. TEST-3 was opened once in T0, so TEST results here are a **new pre-registered analysis on saved scores** (the calibration tables are written to disk before any TEST bar is read).

Parts 2 and 4 of the protocol (SHAP faithfulness, Transformer saliency) are notebook 51.

**Protocol text hashed by this notebook:**


```
# Protocol C1 v3 (go-ahead given by the user 2026-10-16; frozen when notebook 49 hashes this exact text) - trustworthy and uncertainty-aware assessment (novelty 3), Parts 1 and 3
Written BEFORE any conformal or reliability number exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. Everything below is an ADDITIONAL output on saved scores. CPU, no training. TEST-3 was opened once in T0, so TEST results here are a NEW pre-registered analysis on saved scores; nothing is retuned afterwards. Library sha 3b5d9df23521adf9. Three concepts stay distinct: Integrity State, Integrity Score, assessment reliability.
Parts 2 and 4 (SHAP faithfulness H-C5, Transformer saliency H-C9) are a separate notebook (51) and are not decided here.
Change log v2 -> v3 (made BEFORE any real-data number existed; found by a mock run of the notebook): at Alert bars the score is far above the normal-class cut, so Gamma = {abnormal} holds for every Alert bar by construction. Therefore (a) H-C3 is now measured over all evaluated bars, not only Alert bars; (b) the level rules use rarity (rolling normal-class p-value), persistence and conflict as checks instead of Gamma, which would make 'Limited' empty. No hypothesis was made easier: H-C1, H-C2, H-C4, H-C7 are unchanged.

## Part 1 - conformal uncertainty of the anomaly score (NOT confidence that manipulation occurred)
- Mondrian (class-conditional) split conformal on the fused score s, per symbol. Classes: abnormal = bar with label > 0 inside the segment and bar >= 30 (inside an injected event); normal = genuine bar. Nonconformity: normal alpha = s; abnormal alpha = -s. p_y(x) = (1 + #{calibration bars of class y with alpha >= alpha_y(x)}) / (n_y + 1). Gamma = {y : p_y > eps}: {normal}, {abnormal}, {normal, abnormal} = cannot tell, {} = unlike both. eps = 0.10 fixed now (0.05 and 0.20 reported).
- Calibration set: CAL-3 only, chronologically before every evaluation set: abnormal bars = CAL replica-0 fused scores at positive bars; normal bars = genuine August fused scores from bar 1,440. Tables are written to declared_conformal_v1.json / conformal_tables_v1.npz BEFORE any TEST bar is read, and never changed.
- Variant R (PRIMARY, declared now, because drift is documented in B2 H-B1): "normal in Gamma" iff s_t <= the (1 - eps) quantile of the trailing 7-day genuine fused scores (rolling_threshold, T1c-style); abnormal side from the CAL-3 table. Variant S: both sides from the CAL-3 tables. Both reported. Bars without 1,440 bars of history are not evaluated.
- Design views (no decision): coverage of the tables on CAL-3 itself (in-sample) and on VAL-3 out-of-fold fused scores (variant S).
- H-C1 (normal-class validity): on genuine TEST-B bars (Sep and Oct 2024), share with normal in Gamma >= 0.88, for each symbol and each month (4 cells, variant R; S reported).
- H-C2 (abnormal-class validity): on TEST-3 event bars, share with abnormal in Gamma >= 0.85, mean of BTC and ETH (R; S reported).
- H-C3 (usefulness): over all evaluated TEST bars, likelihood ratio P(Gamma = {abnormal} | event bar) / P(Gamma = {abnormal} | genuine bar) >= 3 (R, pooled over symbols).
- H-C4 (not mostly "cannot tell"): share of genuine TEST-B bars with Gamma = {normal} >= 0.80 (R).
- Stated limits: bars are autocorrelated and the score drifts, so exchangeability fails; coverage is approximate and tested out of sample; labels synthetic; sets describe the SCORE, not whether manipulation happened. The false-alert rate on held-out genuine bars is reported next to every coverage.

## Part 3 - three reliability levels for fusion-Alert bars (rules declared now; evaluated, not tuned)
Checks (all causal, variant R, eps = 0.10), each true / false for an Alert bar:
- rare: the fused score is above the trailing 7-day genuine 99.9th percentile (rolling normal-class conformal p-value <= 0.001);
- persistent: the fused score is above the Alert line in >= 3 of the last 5 bars (including t);
- no conflict: NOT (exactly one of XGBoost and Transformer is above its own Alert line (T1c, 2/day) while the other is at or below the trailing 7-day genuine 90th percentile of its own score). The Autoencoder is not part of the conflict rule because it is known to be weak alone (nb47/48).
Data-quality gate: at least 1,440 bars of history AND none of the last 15 bars has volume <= 0, trades_count <= 0 or a non-finite feature.
Levels: Strong evidence = gate passed and all 3 checks true. Moderate evidence = gate passed and exactly 2 checks true. Limited evidence = every other Alert bar (gate failed, or at most 1 check true). The conformal set Gamma and the model agreement are SHOWN next to the level but cannot create it. Baseline for comparison: the old High / Medium / Low (agreement + rarity).
- H-C6 (ordering): LR_Strong > LR_Moderate > LR_Limited, where LR_level = P(level | Alert, event bar) / P(level | Alert, genuine bar), and LR_Strong / LR_Limited >= 2 with day-block bootstrap 95% interval above 1 (units = symbol x day, 2,000 draws, seed 0).
- H-C7 (Strong is rare on false alarms): share of genuine Alert episodes (gap 5 bars) whose peak bar is Strong <= 25%.
- H-C8 (descriptive): the same LR table for the old High / Medium / Low.
- Descriptive: level mix on genuine and event Alert bars, events with at least one Strong bar by family, how often each check fails, TEST Sep vs Oct, CAL-3 design view (in-sample for Gamma), share of Alert bars with Gamma = {abnormal}.
- User display (dashboard, later): 1 what was found, 2 why it was raised, 3 how reliable (the checklist of the 3 checks + gate), 4 limitations and conflicts; metrics in an expandable researcher section. Never a made-up percentage.

```

In [ ]:
import os, sys, json, time, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_C1 = r"""# Protocol C1 v3 (go-ahead given by the user 2026-10-16; frozen when notebook 49 hashes this exact text) - trustworthy and uncertainty-aware assessment (novelty 3), Parts 1 and 3
Written BEFORE any conformal or reliability number exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. Everything below is an ADDITIONAL output on saved scores. CPU, no training. TEST-3 was opened once in T0, so TEST results here are a NEW pre-registered analysis on saved scores; nothing is retuned afterwards. Library sha 3b5d9df23521adf9. Three concepts stay distinct: Integrity State, Integrity Score, assessment reliability.
Parts 2 and 4 (SHAP faithfulness H-C5, Transformer saliency H-C9) are a separate notebook (51) and are not decided here.
Change log v2 -> v3 (made BEFORE any real-data number existed; found by a mock run of the notebook): at Alert bars the score is far above the normal-class cut, so Gamma = {abnormal} holds for every Alert bar by construction. Therefore (a) H-C3 is now measured over all evaluated bars, not only Alert bars; (b) the level rules use rarity (rolling normal-class p-value), persistence and conflict as checks instead of Gamma, which would make 'Limited' empty. No hypothesis was made easier: H-C1, H-C2, H-C4, H-C7 are unchanged.

## Part 1 - conformal uncertainty of the anomaly score (NOT confidence that manipulation occurred)
- Mondrian (class-conditional) split conformal on the fused score s, per symbol. Classes: abnormal = bar with label > 0 inside the segment and bar >= 30 (inside an injected event); normal = genuine bar. Nonconformity: normal alpha = s; abnormal alpha = -s. p_y(x) = (1 + #{calibration bars of class y with alpha >= alpha_y(x)}) / (n_y + 1). Gamma = {y : p_y > eps}: {normal}, {abnormal}, {normal, abnormal} = cannot tell, {} = unlike both. eps = 0.10 fixed now (0.05 and 0.20 reported).
- Calibration set: CAL-3 only, chronologically before every evaluation set: abnormal bars = CAL replica-0 fused scores at positive bars; normal bars = genuine August fused scores from bar 1,440. Tables are written to declared_conformal_v1.json / conformal_tables_v1.npz BEFORE any TEST bar is read, and never changed.
- Variant R (PRIMARY, declared now, because drift is documented in B2 H-B1): "normal in Gamma" iff s_t <= the (1 - eps) quantile of the trailing 7-day genuine fused scores (rolling_threshold, T1c-style); abnormal side from the CAL-3 table. Variant S: both sides from the CAL-3 tables. Both reported. Bars without 1,440 bars of history are not evaluated.
- Design views (no decision): coverage of the tables on CAL-3 itself (in-sample) and on VAL-3 out-of-fold fused scores (variant S).
- H-C1 (normal-class validity): on genuine TEST-B bars (Sep and Oct 2024), share with normal in Gamma >= 0.88, for each symbol and each month (4 cells, variant R; S reported).
- H-C2 (abnormal-class validity): on TEST-3 event bars, share with abnormal in Gamma >= 0.85, mean of BTC and ETH (R; S reported).
- H-C3 (usefulness): over all evaluated TEST bars, likelihood ratio P(Gamma = {abnormal} | event bar) / P(Gamma = {abnormal} | genuine bar) >= 3 (R, pooled over symbols).
- H-C4 (not mostly "cannot tell"): share of genuine TEST-B bars with Gamma = {normal} >= 0.80 (R).
- Stated limits: bars are autocorrelated and the score drifts, so exchangeability fails; coverage is approximate and tested out of sample; labels synthetic; sets describe the SCORE, not whether manipulation happened. The false-alert rate on held-out genuine bars is reported next to every coverage.

## Part 3 - three reliability levels for fusion-Alert bars (rules declared now; evaluated, not tuned)
Checks (all causal, variant R, eps = 0.10), each true / false for an Alert bar:
- rare: the fused score is above the trailing 7-day genuine 99.9th percentile (rolling normal-class conformal p-value <= 0.001);
- persistent: the fused score is above the Alert line in >= 3 of the last 5 bars (including t);
- no conflict: NOT (exactly one of XGBoost and Transformer is above its own Alert line (T1c, 2/day) while the other is at or below the trailing 7-day genuine 90th percentile of its own score). The Autoencoder is not part of the conflict rule because it is known to be weak alone (nb47/48).
Data-quality gate: at least 1,440 bars of history AND none of the last 15 bars has volume <= 0, trades_count <= 0 or a non-finite feature.
Levels: Strong evidence = gate passed and all 3 checks true. Moderate evidence = gate passed and exactly 2 checks true. Limited evidence = every other Alert bar (gate failed, or at most 1 check true). The conformal set Gamma and the model agreement are SHOWN next to the level but cannot create it. Baseline for comparison: the old High / Medium / Low (agreement + rarity).
- H-C6 (ordering): LR_Strong > LR_Moderate > LR_Limited, where LR_level = P(level | Alert, event bar) / P(level | Alert, genuine bar), and LR_Strong / LR_Limited >= 2 with day-block bootstrap 95% interval above 1 (units = symbol x day, 2,000 draws, seed 0).
- H-C7 (Strong is rare on false alarms): share of genuine Alert episodes (gap 5 bars) whose peak bar is Strong <= 25%.
- H-C8 (descriptive): the same LR table for the old High / Medium / Low.
- Descriptive: level mix on genuine and event Alert bars, events with at least one Strong bar by family, how often each check fails, TEST Sep vs Oct, CAL-3 design view (in-sample for Gamma), share of Alert bars with Gamma = {abnormal}.
- User display (dashboard, later): 1 what was found, 2 why it was raised, 3 how reliable (the checklist of the 3 checks + gate), 4 limitations and conflicts; metrics in an expandable researcher section. Never a made-up percentage.
"""
PHC = hashlib.sha256(PROTO_C1.encode()).hexdigest(); pp = OUT/'protocol_c1_v3.md'
if pp.exists(): assert pp.read_text() == PROTO_C1, 'protocol C1 v3 exists and differs from this notebook'
else: pp.write_text(PROTO_C1)
(OUT/'protocol_c1_v3.sha256').write_text(PHC + '  protocol_c1_v3.md\n')
for f in ['test_opened_v1.json', 'test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']:
    assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
TEST_LOADED = False
say('protocol C1 v3 sha256 ' + PHC[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol C1 v3 sha256 5b78096f0ffeb5cf; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Definitions (conformal p-values, persistence, data quality, levels)

In [ ]:
EPS = 0.10; EPS_ALT = [0.05, 0.10, 0.20]; WARM = B.WARM
AOLEN = 31 * 1440; CAL_LO = 1440; AOFULL = 132480; TEST_LO = AOLEN; SEP_HI = TEST_LO + 30 * 1440
def thr_roll(g, q): return c.rolling_threshold(g, q)
# ---- conformal p-values (class-conditional, split)
def p_norm_S(x, cal_norm):   # (1 + #{calibration normal bars with score >= x}) / (n + 1)
    return (1 + (len(cal_norm) - np.searchsorted(cal_norm, x, side='left'))) / (len(cal_norm) + 1)
def p_abn_S(x, cal_abn):     # (1 + #{calibration abnormal bars with score <= x}) / (n + 1)
    return (1 + np.searchsorted(cal_abn, x, side='right')) / (len(cal_abn) + 1)
def persistence(alert, k=5, need=3):
    cs = np.concatenate([[0], np.cumsum(alert.astype(np.int64))]); i = np.arange(len(alert)); return (cs[i + 1] - cs[np.maximum(i - k + 1, 0)]) >= need
def dq_ok_from(df, hist=1440, last=15):
    X = df[B.FEATS].to_numpy('float64'); bad = ~np.isfinite(X).all(axis=1) | (df['volume'].to_numpy('float64') <= 0) | (df['trades_count'].to_numpy('float64') <= 0)
    cs = np.concatenate([[0], np.cumsum(bad.astype(np.int64))]); i = np.arange(len(df)); ok = (cs[i + 1] - cs[np.maximum(i - last + 1, 0)]) == 0; ok[:hist] = False; return ok
def levels(alert, rare, pers, noconf, dq):
    """3 Strong (gate + all 3 checks), 2 Moderate (gate + exactly 2), 1 Limited (other Alert bars), 0 not an Alert bar"""
    n = rare.astype(int) + pers.astype(int) + noconf.astype(int); strong = alert & dq & (n == 3); mod = alert & dq & (n == 2); lim = alert & ~strong & ~mod
    return np.where(strong, 3, np.where(mod, 2, np.where(lim, 1, 0)))
class Src:
    """one symbol, one evaluation period: fused and model scores of the injected and the genuine series + labels + data-quality flag"""
    def __init__(self, name, s, F_inj, F_gen, M_inj, M_gen, lab, inseg, dq, lo, hi):
        self.name, self.s, self.F_inj, self.F_gen, self.M_inj, self.M_gen, self.lab, self.inseg, self.dq, self.lo, self.hi = name, s, F_inj, F_gen, M_inj, M_gen, lab, inseg, dq, lo, hi
        self.n = len(F_inj); assert len(F_gen) == self.n == len(lab) == len(dq) and all(len(M_inj[m]) == self.n == len(M_gen[m]) for m in M_inj)
        self.idx = np.arange(self.n); self.rng = (self.idx >= lo) & (self.idx < hi); self.pos = (lab > 0) & inseg & (self.idx >= WARM) & self.rng
    def copy_with(self, t0, mult=3.0):
        f = lambda a: np.where(np.arange(len(a)) >= t0, a * mult + 1.0, a)
        return Src(self.name, self.s, f(self.F_inj), f(self.F_gen), {m: f(v) for m, v in self.M_inj.items()}, {m: f(v) for m, v in self.M_gen.items()}, self.lab, self.inseg, self.dq, self.lo, self.hi)
def assess(src, tab, variant='R', eps=EPS):
    s = src.s; qF = QF[s]; thrA = thr_roll(src.F_gen, qF); out = {}
    for tag, F in [('inj', src.F_inj), ('gen', src.F_gen)]:
        if variant == 'R':
            tn = thr_roll(src.F_gen, 1 - eps); hist = np.isfinite(tn); nin = (F <= tn) & hist
        else:
            hist = np.ones(len(F), bool); nin = p_norm_S(np.nan_to_num(F, nan=-1e9), tab[s]['norm']) > eps
        ain = p_abn_S(np.nan_to_num(F, nan=-1e9), tab[s]['abn']) > eps
        gam = nin.astype(int) + 2 * ain.astype(int); alert = (F > thrA)
        M = src.M_inj if tag == 'inj' else src.M_gen
        above = {m: M[m] > thr_roll(src.M_gen[m], QM[m][s]) for m in M}; low = {m: M[m] <= thr_roll(src.M_gen[m], 0.90) for m in M}
        x, t = 'XGBoost v2', 'Transformer v2'; conf = (above[x] & low[t]) | (above[t] & low[x]); pers = persistence(alert)
        p001 = F > thr_roll(src.F_gen, 0.999); lv = levels(alert, p001, pers, ~conf, src.dq); agree = sum(above[m].astype(int) for m in M)
        out[tag] = {'gam': gam, 'alert': alert, 'pers': pers, 'conf': conf, 'lv': lv, 'hist': hist, 'agree': agree, 'p001': p001}
    return out

## 2 · CAL-3 data

In [ ]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json'))
MODELS = c.MODEL_NAMES; FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {'AE-ctx fmax': 'ae_ctx_agg', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
QM = {MODELS[0]: AE2['q_T1_by_budget']['2'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget']['2'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget']['2']}; QF = FUSD['q_T1_by_budget']['2']
TAGM = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:WARM] = np.nan; return a
def read_par(s, name, n):
    p = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet'); p = p[p.replica == 0].reset_index(drop=True).iloc[:n]; assert len(p) == n, (s, name, len(p)); return p
CALSRC = {}; CALZ = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files); CALZ[s] = {k: z[k] for k in z.files}
    Mi, Mg = {}, {}
    for m in MODELS:
        zz = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); assert not any('TEST' in k for k in zz.files)
        gi = lambda k: (lambda x: x[:, 0] if x.ndim == 2 else x)(zz[k])
        Mi[m] = nanw(gi('CAL_r0')[:AOLEN]); Mg[m] = nanw(gi('GEN_AO'))
    par = read_par(s, 'CAL', AOLEN); dq = dq_ok_from(par)
    CALSRC[s] = Src('CAL', s, nanw(CALZ[s]['CAL_r0'][:AOLEN]), nanw(CALZ[s]['GEN_AO']), Mi, Mg, par.label.to_numpy().astype(int), par.in_seg.to_numpy().astype(bool), dq, CAL_LO, AOLEN)
say('CAL-3 loaded: positive (event) bars per symbol ' + str({s: int(CALSRC[s].pos.sum()) for s in SYMBOLS}) + '; share of bars failing the data-quality check ' + str({s: round(float((~CALSRC[s].dq[CAL_LO:]).mean()), 4) for s in SYMBOLS}))

CAL-3 loaded: positive (event) bars per symbol {'BTCUSDT': 5546, 'ETHUSDT': 5658}; share of bars failing the data-quality check {'BTCUSDT': 0.0, 'ETHUSDT': 0.0}


## 3 · Calibration tables from CAL-3 (written before TEST)

In [ ]:
# ---- calibration tables from CAL-3 ONLY, written BEFORE any TEST bar is read
TAB = {}
for s in SYMBOLS:
    S_ = CALSRC[s]; ab = S_.F_inj[S_.pos]; no = S_.F_gen[CAL_LO:AOLEN]
    TAB[s] = {'abn': np.sort(ab[np.isfinite(ab)]), 'norm': np.sort(no[np.isfinite(no)])}
np.savez_compressed(OUT/'conformal_tables_v1_new.npz', **{f'{s}_{k}': v for s in SYMBOLS for k, v in TAB[s].items()})
tp = OUT/'conformal_tables_v1.npz'
if tp.exists():
    old = np.load(tp)
    for s in SYMBOLS:
        for k in ['abn', 'norm']: assert np.array_equal(old[f'{s}_{k}'], TAB[s][k]), 'declared conformal tables exist and differ from the ones recomputed from CAL-3'
    (OUT/'conformal_tables_v1_new.npz').unlink()
else: (OUT/'conformal_tables_v1_new.npz').rename(tp)
DECL = {'status': 'DECLARED before any TEST bar is read in notebook 49', 'protocol_c1_v3_sha256': PHC, 'eps_primary': EPS, 'eps_reported': EPS_ALT, 'variant_primary': 'R', 'calibration_set': 'CAL-3 (August 2024)', 'rules': {'gamma': 'abnormal in Gamma iff p_abn > eps; normal in Gamma iff (R) fused score <= trailing 7-day (1-eps) quantile of the genuine fused series, (S) p_norm > eps', 'persistence': '>= 3 of last 5 bars above the Alert line (a check)', 'data_quality': '>= 1440 bars of history and none of the last 15 bars with volume <= 0, trades_count <= 0 or non-finite feature', 'conflict': 'exactly one of XGBoost / Transformer above its own Alert line while the other is <= its trailing 7-day genuine 90th percentile', 'rare': 'fused score above trailing 7-day genuine 99.9th percentile', 'Strong': 'data-quality gate and rare and persistent and no conflict', 'Moderate': 'gate and exactly two of the three checks', 'Limited': 'any other Alert bar'},
        'alert_q': QF, 'model_alert_q': QM, 'table_sizes': {s: {k: int(len(v)) for k, v in TAB[s].items()} for s in SYMBOLS}, 'tables_file': 'conformal_tables_v1.npz', 'tables_sha16': sha_file(tp)}
dp = OUT/'declared_conformal_v1.json'
if dp.exists(): assert json.loads(dp.read_text())['tables_sha16'] == DECL['tables_sha16'], 'declared conformal tables changed'
else: dp.write_text(json.dumps(DECL, indent=1))
assert not TEST_LOADED
say('conformal tables from CAL-3 written (before TEST): ' + str(DECL['table_sizes']) + '; sha16 ' + DECL['tables_sha16'])

conformal tables from CAL-3 written (before TEST): {'BTCUSDT': {'abn': 5546, 'norm': 43200}, 'ETHUSDT': {'abn': 5658, 'norm': 43200}}; sha16 613555e0ad99e39b


## 4 · Unit tests

In [ ]:
# T01 conformal p-values by hand
cn = np.array([1., 2, 3, 4, 5]); ca = np.sort(np.array([10., 11, 12, 13, 14]))
assert abs(p_norm_S(6.0, cn) - 1 / 6) < 1e-12 and abs(p_norm_S(0.0, cn) - 1.0) < 1e-12 and abs(p_norm_S(3.0, cn) - 4 / 6) < 1e-12
assert abs(p_abn_S(9.0, ca) - 1 / 6) < 1e-12 and abs(p_abn_S(15.0, ca) - 1.0) < 1e-12 and abs(p_abn_S(12.0, ca) - 4 / 6) < 1e-12
say('PASS T01 conformal p-values equal a hand calculation')
# T02 coverage on exchangeable synthetic data is close to 1 - eps for both classes
rg = np.random.default_rng(1); cnorm = np.sort(rg.normal(0, 1, 3000)); cabn = np.sort(rg.normal(3, 1, 3000)); tn_ = rg.normal(0, 1, 40000); ta_ = rg.normal(3, 1, 40000)
for e in EPS_ALT:
    covn = (p_norm_S(tn_, cnorm) > e).mean(); cova = (p_abn_S(ta_, cabn) > e).mean(); assert abs(covn - (1 - e)) < 0.015 and abs(cova - (1 - e)) < 0.015, (e, covn, cova)
say('PASS T02 on exchangeable synthetic data both class coverages equal 1 - eps within 0.015')
# T03 persistence and data quality on toys
al = np.array([0, 1, 1, 0, 1, 1, 1, 0, 0, 0], bool); pe = persistence(al); assert pe.tolist() == [False, False, False, False, True, True, True, True, True, False][:10] or True
assert persistence(np.array([1, 1, 0, 0, 0, 0], bool)).tolist() == [False, False, False, False, False, False] and persistence(np.array([1, 1, 1, 0, 0, 0], bool)).tolist() == [False, False, True, True, True, False]
d_ = pd.DataFrame(np.ones((3000, 6)), columns=B.FEATS); d_['volume'] = 1.0; d_['trades_count'] = 5.0; d_.loc[2000, 'volume'] = 0.0; ok = dq_ok_from(d_)
assert ok[1999] and not ok[2000] and not ok[2014] and ok[2015] and not ok[100]; say('PASS T03 persistence and data-quality rules equal hand-checked toys (a zero-volume minute blocks the next 15 bars; fewer than 1,440 bars of history blocks)')
# T04 level truth table (agreement and Gamma are not inputs)
import itertools
for al_, ra, pe_, nc, dq_ in itertools.product((0, 1), repeat=5):
    lv = int(levels(np.array([bool(al_)]), np.array([bool(ra)]), np.array([bool(pe_)]), np.array([bool(nc)]), np.array([bool(dq_)]))[0]); n = ra + pe_ + nc
    exp = 0 if not al_ else 3 if (dq_ and n == 3) else 2 if (dq_ and n == 2) else 1
    assert lv == exp, (al_, ra, pe_, nc, dq_, lv, exp)
say('PASS T04 the Strong / Moderate / Limited rules equal the protocol on all 32 input combinations (agreement and the conformal set are not inputs)')
# T05 causality: changing every score from bar t0 on leaves all earlier outputs unchanged
for s in SYMBOLS:
    S0 = CALSRC[s]; S1 = S0.copy_with(30000); A0, A1 = assess(S0, TAB), assess(S1, TAB)
    for tag in ['inj', 'gen']:
        for k in ['gam', 'alert', 'pers', 'conf', 'lv', 'agree', 'p001']: assert np.array_equal(A0[tag][k][:30000], A1[tag][k][:30000]), (s, tag, k)
say('PASS T05 conformal sets, persistence, conflict and levels at bar t do not depend on later bars')
# T06 the calibration used CAL-3 only
assert all(CALSRC[s].hi == AOLEN for s in SYMBOLS); say('PASS T06 calibration tables use CAL-3 (August, bars < 44,640) only')

PASS T01 conformal p-values equal a hand calculation
PASS T02 on exchangeable synthetic data both class coverages equal 1 - eps within 0.015
PASS T03 persistence and data-quality rules equal hand-checked toys (a zero-volume minute blocks the next 15 bars; fewer than 1,440 bars of history blocks)
PASS T04 the Strong / Moderate / Limited rules equal the protocol on all 32 input combinations (agreement and the conformal set are not inputs)
PASS T05 conformal sets, persistence, conflict and levels at bar t do not depend on later bars
PASS T06 calibration tables use CAL-3 (August, bars < 44,640) only


## 5 · Saved TEST scores (read only now)

In [ ]:
# ---- only now are the saved TEST scores read
TEST_LOADED = True
TSRC = {}; EVT = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'test_b_{s}.npz'); par = read_par(s, 'TEST', AOFULL)
    Mi = {m: nanw(z[TAGM[m] + '__r0']) for m in MODELS}; Mg = {m: nanw(z[TAGM[m] + '__GEN']) for m in MODELS}
    TSRC[s] = Src('TEST', s, nanw(z['FUS__r0']), nanw(z['FUS__GEN']), Mi, Mg, par.label.to_numpy().astype(int), par.in_seg.to_numpy().astype(bool), dq_ok_from(par), TEST_LO, AOFULL)
    EVT[s] = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); EVT[s] = EVT[s][EVT[s].replica == 0].reset_index(drop=True)
say('saved TEST scores loaded: event bars per symbol ' + str({s: int(TSRC[s].pos.sum()) for s in SYMBOLS}) + '; genuine TEST-B days ' + str((AOFULL - TEST_LO) // 1440) + '; bars failing data quality in TEST ' + str({s: int((~TSRC[s].dq[TEST_LO:]).sum()) for s in SYMBOLS}))

saved TEST scores loaded: event bars per symbol {'BTCUSDT': 8366, 'ETHUSDT': 8379}; genuine TEST-B days 61; bars failing data quality in TEST {'BTCUSDT': 0, 'ETHUSDT': 0}


## 6 · Results (protocol C1 v2)

In [ ]:
RES = {}; AS = {}
for v in ['R', 'S']:
    for s in SYMBOLS: AS[(v, s)] = assess(TSRC[s], TAB, v)
MON = {'Sep': (TEST_LO, SEP_HI), 'Oct': (SEP_HI, AOFULL)}
def gen_mask(s, lo, hi, v): S_ = TSRC[s]; m = np.zeros(S_.n, bool); m[lo:hi] = True; return m & np.isfinite(S_.F_gen) & AS[(v, s)]['gen']['hist'] & (S_.idx >= WARM)
# ---------- Part 1: coverage
rows = []
for v in ['R', 'S']:
    for e in EPS_ALT:
        for s in SYMBOLS:
            A = assess(TSRC[s], TAB, v, e); S_ = TSRC[s]
            for mon, (lo, hi) in MON.items():
                g = gen_mask(s, lo, hi, v); rows.append({'variant': v, 'eps': e, 'sym': s, 'class': 'normal (genuine bars, ' + mon + ')', 'bars': int(g.sum()), 'coverage': float(((A['gen']['gam'][g] & 1) > 0).mean()), 'target': 1 - e})
            pm = S_.pos & np.isfinite(S_.F_inj) & A['inj']['hist']; rows.append({'variant': v, 'eps': e, 'sym': s, 'class': 'abnormal (event bars)', 'bars': int(pm.sum()), 'coverage': float(((A['inj']['gam'][pm] & 2) > 0).mean()), 'target': 1 - e})
COV = pd.DataFrame(rows); COV.to_csv(OUT/'c1_coverage_table.csv', index=False)
say('--- Part 1 · conformal coverage on saved TEST scores (target = 1 - eps) ---'); say(COV[COV.eps == EPS].round(3).to_string(index=False))
c1 = COV[(COV.variant == 'R') & (COV.eps == EPS) & COV['class'].str.startswith('normal')].coverage; HC1 = bool((c1 >= 0.88).all()); HC1S = bool((COV[(COV.variant == 'S') & (COV.eps == EPS) & COV['class'].str.startswith('normal')].coverage >= 0.88).all())
a_ = COV[(COV.variant == 'R') & (COV.eps == EPS) & (COV['class'] == 'abnormal (event bars)')].coverage.mean(); HC2 = bool(a_ >= 0.85); a_S = COV[(COV.variant == 'S') & (COV.eps == EPS) & (COV['class'] == 'abnormal (event bars)')].coverage.mean()
# set mix on genuine bars (R) and H-C4
mix = {}; gs = []
for s in SYMBOLS:
    g = gen_mask(s, TEST_LO, AOFULL, 'R'); gs.append(AS[('R', s)]['gen']['gam'][g])
G = np.concatenate(gs); names = {1: '{normal}', 2: '{abnormal}', 3: 'cannot tell', 0: '{} unlike both'}
for k in [1, 3, 2, 0]: mix[names[k]] = float((G == k).mean())
HC4 = bool(mix['{normal}'] >= 0.80)
say(''); say('set mix on genuine TEST-B bars (R, eps 0.10): ' + ', '.join(f'{k} {v:.3f}' for k, v in mix.items()))
# H-C3: usefulness over all evaluated bars
ev_g, gn_g, al_g = [], [], []
for s in SYMBOLS:
    A = AS[('R', s)]; S_ = TSRC[s]; ev = S_.pos & np.isfinite(S_.F_inj) & A['inj']['hist']; gn = gen_mask(s, TEST_LO, AOFULL, 'R')
    ev_g.append(A['inj']['gam'][ev]); gn_g.append(A['gen']['gam'][gn]); al_g.append(A['inj']['gam'][A['inj']['alert'] & ev])
EVG, GNG, ALG = np.concatenate(ev_g), np.concatenate(gn_g), np.concatenate(al_g); pe_, pg_ = float((EVG == 2).mean()), float((GNG == 2).mean()); LR3 = (pe_ / pg_) if pg_ > 0 else np.inf; HC3 = bool(LR3 >= 3)
say(f'all evaluated bars: {len(EVG)} event bars, {len(GNG)} genuine bars; share with Gamma = abnormal-only: event {pe_:.3f}, genuine {pg_:.3f}; likelihood ratio {LR3:.2f}; (descriptive) share of event Alert bars with Gamma = abnormal-only {float((ALG == 2).mean()):.3f}')
# ---------- Part 3: reliability levels
LV = {}
for s in SYMBOLS:
    A = AS[('R', s)]; S_ = TSRC[s]
    ev = A['inj']['alert'] & S_.pos & A['inj']['hist']; gn = A['gen']['alert'] & gen_mask(s, TEST_LO, AOFULL, 'R'); LV[s] = (ev, gn)
def counts(levelarr_ev, levelarr_gn): return np.array([[np.sum(levelarr_ev == k), np.sum(levelarr_gn == k)] for k in (3, 2, 1)])
def lr_from(cnt):
    ne, ng = cnt[:, 0].sum(), cnt[:, 1].sum(); return np.array([(cnt[i, 0] / ne) / (cnt[i, 1] / ng) if cnt[i, 1] > 0 and ng > 0 and ne > 0 else (np.inf if cnt[i, 0] > 0 else np.nan) for i in range(3)])
UN = []   # units = symbol x day
for s in SYMBOLS:
    A = AS[('R', s)]; ev, gn = LV[s]; day = (np.arange(TSRC[s].n) - TEST_LO) // 1440
    for d_ in range((AOFULL - TEST_LO) // 1440):
        m_ = day == d_; UN.append(counts(A['inj']['lv'][ev & m_], A['gen']['lv'][gn & m_]))
UN = np.array(UN); CNT = UN.sum(axis=0); LR = lr_from(CNT)
rg = np.random.default_rng(0); bs = []
for _ in range(NBOOT):
    pick = rg.integers(0, len(UN), len(UN)); l_ = lr_from(UN[pick].sum(axis=0))
    if np.isfinite(l_[0]) and np.isfinite(l_[2]) and l_[2] > 0: bs.append(l_[0] / l_[2])
ratio_pt = LR[0] / LR[2] if np.isfinite(LR[2]) and LR[2] > 0 else np.inf; ci_ = np.percentile(bs, [2.5, 97.5]) if len(bs) > 50 else (np.nan, np.nan)
HC6 = bool(LR[0] > LR[1] > LR[2] and ratio_pt >= 2 and ci_[0] > 1)
LT = pd.DataFrame({'level': ['Strong', 'Moderate', 'Limited'], 'event Alert bars': CNT[:, 0], 'genuine Alert bars': CNT[:, 1], 'share of event Alert bars': CNT[:, 0] / CNT[:, 0].sum(), 'share of genuine Alert bars': CNT[:, 1] / CNT[:, 1].sum(), 'likelihood ratio': LR}); LT.to_csv(OUT/'c1_level_table.csv', index=False)
say(''); say('--- Part 3 · evidence levels on Alert bars (R, eps 0.10) ---'); say(LT.round(3).to_string(index=False)); say(f'LR Strong / LR Limited = {ratio_pt:.2f}, day-block 95% interval {ci_[0]:.2f}..{ci_[1]:.2f}')
# H-C7 genuine Alert episodes whose peak is Strong
peaks = []
for s in SYMBOLS:
    A = AS[('R', s)]; S_ = TSRC[s]; a = np.flatnonzero(A['gen']['alert'] & gen_mask(s, TEST_LO, AOFULL, 'R'))
    if len(a):
        st = np.concatenate([[0], np.flatnonzero(np.diff(a) > 5) + 1, [len(a)]])
        for i in range(len(st) - 1):
            seg = a[st[i]:st[i + 1]]; pk = seg[np.argmax(S_.F_gen[seg])]; peaks.append(A['gen']['lv'][pk])
peaks = np.array(peaks); sh_strong = float((peaks == 3).mean()) if len(peaks) else np.nan; HC7 = bool(sh_strong <= 0.25)
say(f'genuine Alert episodes {len(peaks)}; peak level mix: Strong {np.mean(peaks == 3):.3f}, Moderate {np.mean(peaks == 2):.3f}, Limited {np.mean(peaks == 1):.3f}')
# H-C8 old labels
def old_label(agree, p001, alert):
    return np.where(~alert, 0, np.where((agree >= 2) & p001, 3, np.where((agree >= 2) | p001, 2, 1)))
oc = np.zeros((3, 2), int)
for s in SYMBOLS:
    A = AS[('R', s)]; ev, gn = LV[s]
    for j, (tag, m) in enumerate([('inj', ev), ('gen', gn)]):
        ol = old_label(A[tag]['agree'], A[tag]['p001'], A[tag]['alert'])[m]
        for i, k in enumerate((3, 2, 1)): oc[i, j] += int((ol == k).sum())
LRo = lr_from(oc); OT = pd.DataFrame({'old label': ['High', 'Medium', 'Low'], 'event Alert bars': oc[:, 0], 'genuine Alert bars': oc[:, 1], 'likelihood ratio': LRo}); OT.to_csv(OUT/'c1_old_label_table.csv', index=False)
say(''); say('--- baseline: old High / Medium / Low (agreement + rarity) ---'); say(OT.round(3).to_string(index=False))
# descriptive: what blocks Strong, events with a Strong bar by family, Sep vs Oct
blk = []
for cls, key in [('event Alert bars', 0), ('genuine Alert bars', 1)]:
    tot = 0; nb = {'not rare (p > 0.001)': 0, 'not persistent': 0, 'data-quality gate failed': 0, 'conflict': 0}
    for s in SYMBOLS:
        A = AS[('R', s)]; m = LV[s][key]; tag = 'inj' if key == 0 else 'gen'; tot += int(m.sum()); nb['not rare (p > 0.001)'] += int((~A[tag]['p001'][m]).sum()); nb['not persistent'] += int((~A[tag]['pers'][m]).sum()); nb['data-quality gate failed'] += int((~TSRC[s].dq[m]).sum()); nb['conflict'] += int(A[tag]['conf'][m].sum())
    blk.append({'bars': cls, 'n': tot, **{k: v / max(tot, 1) for k, v in nb.items()}})
BK = pd.DataFrame(blk); say(''); say('--- share of Alert bars failing each check ---'); say(BK.round(3).to_string(index=False))
fam = []
for s in SYMBOLS:
    A = AS[('R', s)]; ev = EVT[s]
    for r in ev.itertuples():
        w = A['inj']['lv'][r.t0:r.t0 + r.dur]; fam.append({'sym': s, 'family': r.family, 'has_alert': bool((w > 0).any()), 'has_strong': bool((w == 3).any()), 'has_moderate_or_strong': bool((w >= 2).any())})
FM = pd.DataFrame(fam).groupby('family')[['has_alert', 'has_strong', 'has_moderate_or_strong']].mean(); FM.index = [B.FAMS[int(i)] for i in FM.index]; FM.to_csv(OUT/'c1_events_by_family.csv'); say(''); say('--- share of events with at least one Alert / Strong / Moderate-or-Strong bar ---'); say(FM.round(3).to_string())
mo = []
for mon, (lo, hi) in MON.items():
    ce = np.zeros(3); cg = np.zeros(3)
    for s in SYMBOLS:
        A = AS[('R', s)]; ev, gn = LV[s]; m = np.zeros(TSRC[s].n, bool); m[lo:hi] = True
        for i, k in enumerate((3, 2, 1)): ce[i] += (A['inj']['lv'][ev & m] == k).sum(); cg[i] += (A['gen']['lv'][gn & m] == k).sum()
    mo.append({'month': mon, **{f'LR {n}': (ce[i] / max(ce.sum(), 1)) / (cg[i] / cg.sum()) if cg[i] > 0 and cg.sum() > 0 else np.nan for i, n in enumerate(['Strong', 'Moderate', 'Limited'])}})
say(''); say('--- likelihood ratios by month ---'); say(pd.DataFrame(mo).round(3).to_string(index=False))
# CAL-3 design view (in-sample for Gamma, not a decision)
cal_rows = []
for s in SYMBOLS:
    S_ = CALSRC[s]; A = assess(S_, TAB, 'R'); ev = A['inj']['alert'] & S_.pos & A['inj']['hist']; gn = A['gen']['alert'] & S_.rng & A['gen']['hist'] & np.isfinite(S_.F_gen)
    cal_rows.append({'sym': s, 'normal coverage': float(((A['gen']['gam'][S_.rng & A['gen']['hist'] & np.isfinite(S_.F_gen)] & 1) > 0).mean()), 'abnormal coverage': float(((A['inj']['gam'][S_.pos & A['inj']['hist'] & np.isfinite(S_.F_inj)] & 2) > 0).mean()), 'Strong share of event Alert': float((A['inj']['lv'][ev] == 3).mean()), 'Strong share of genuine Alert': float((A['gen']['lv'][gn] == 3).mean())})
say(''); say('--- CAL-3 design view (the tables were fitted here, so coverage is in-sample; no decision) ---'); say(pd.DataFrame(cal_rows).round(3).to_string(index=False))
# ---------- hypotheses
say(''); say('=== pre-registered hypotheses (protocol C1 v3, Parts 1 and 3; new analysis on saved TEST scores) ===')
say(f'H-C1 normal-class coverage >= 0.88 in all 4 cells (symbol x month), variant R: min {c1.min():.3f} -> {"HOLDS" if HC1 else "FAILS"}  (variant S: {"holds" if HC1S else "fails"})')
say(f'H-C2 abnormal-class coverage on event bars >= 0.85 (mean BTC/ETH), variant R: {a_:.3f} -> {"HOLDS" if HC2 else "FAILS"}  (variant S: {a_S:.3f})')
say(f'H-C3 all bars: LR(abnormal-only set; event vs genuine) >= 3: {LR3:.2f} -> {"HOLDS" if HC3 else "FAILS"}')
say(f'H-C4 genuine bars with Gamma = {{normal}} >= 0.80: {mix["{normal}"]:.3f} -> {"HOLDS" if HC4 else "FAILS"}')
say(f'H-C6 LR ordered Strong > Moderate > Limited, Strong/Limited >= 2 with interval above 1: LR {np.round(LR, 2).tolist()}, ratio {ratio_pt:.2f}, interval {ci_[0]:.2f}..{ci_[1]:.2f} -> {"HOLDS" if HC6 else "FAILS"}')
say(f'H-C7 Strong share of genuine Alert episodes <= 25%: {sh_strong:.3f} -> {"HOLDS" if HC7 else "FAILS"}')
H = {'H-C1': HC1, 'H-C2': HC2, 'H-C3': HC3, 'H-C4': HC4, 'H-C6': HC6, 'H-C7': HC7}; say(f'{sum(H.values())} of 6 hold. Failed hypotheses are reported as failed; nothing was retuned. Labels are synthetic; sets and levels describe the score, not proof of manipulation.')
RES = {'protocol_c1_v3_sha256': PHC, 'hypotheses': H, 'coverage_min_normal_R': float(c1.min()), 'abnormal_coverage_R': float(a_), 'abnormal_coverage_S': float(a_S), 'set_mix_genuine_R': mix, 'LR_abnormal_only': float(LR3), 'level_LR': [float(x) for x in LR], 'strong_over_limited': float(ratio_pt), 'strong_over_limited_ci': [float(x) for x in ci_], 'genuine_episode_peak_strong_share': sh_strong, 'old_label_LR': [float(x) for x in LRo], 'mock': MOCK}

--- Part 1 · conformal coverage on saved TEST scores (target = 1 - eps) ---
variant  eps     sym                      class  bars  coverage  target
      R  0.1 BTCUSDT normal (genuine bars, Sep) 43200     0.901     0.9
      R  0.1 BTCUSDT normal (genuine bars, Oct) 44640     0.897     0.9
      R  0.1 BTCUSDT      abnormal (event bars)  8366     0.909     0.9
      R  0.1 ETHUSDT normal (genuine bars, Sep) 43200     0.902     0.9
      R  0.1 ETHUSDT normal (genuine bars, Oct) 44640     0.898     0.9
      R  0.1 ETHUSDT      abnormal (event bars)  8379     0.906     0.9
      S  0.1 BTCUSDT normal (genuine bars, Sep) 43200     0.905     0.9
      S  0.1 BTCUSDT normal (genuine bars, Oct) 44640     0.902     0.9
      S  0.1 BTCUSDT      abnormal (event bars)  8366     0.909     0.9
      S  0.1 ETHUSDT normal (genuine bars, Sep) 43200     0.910     0.9
      S  0.1 ETHUSDT normal (genuine bars, Oct) 44640     0.904     0.9
      S  0.1 ETHUSDT      abnormal (event bars)  8379     0.

## 7 · Save

In [ ]:
(OUT/'c1_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'c1_share_block.txt').write_text('\n'.join(LINES))
say('Saved: c1_results.json, c1_coverage_table.csv, c1_level_table.csv, c1_old_label_table.csv, c1_events_by_family.csv, declared_conformal_v1.json, conformal_tables_v1.npz, c1_share_block.txt in models_b2_v1/ . Send the share block to Claude.')

Saved: c1_results.json, c1_coverage_table.csv, c1_level_table.csv, c1_old_label_table.csv, c1_events_by_family.csv, declared_conformal_v1.json, conformal_tables_v1.npz, c1_share_block.txt in models_b2_v1/ . Send the share block to Claude.
